# Stock Return Prediction using Triple-Barrier Labeling

## 0. Imports

In [ ]:
import os
import numpy as np
from typing import Optional, Tuple, List, Dict

import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

## 1. Label Construction

### Configuration

In [33]:
# --------------------------------------------------
# Config
# --------------------------------------------------
DATA_DIR = "./data"
PANEL_FILE = os.path.join(DATA_DIR, "vn30_panel_daily.csv")
PROCESSED_OUT = os.path.join(DATA_DIR, "processed_vn30_panel_daily.csv")

START = "2018-01-01"
END   = "2026-03-15"

# Triple-barrier parameters
VOL_LOOKBACK = 20          # rolling vol window
VERTICAL_HORIZON = 10      # number of trading days forward
PT_MULT = 2.0              # profit-taking barrier = + PT_MULT * vol
SL_MULT = 2.0              # stop-loss barrier     = - SL_MULT * vol

### Load raw panel data

In [21]:
# --------------------------------------------------
# Load panel
# --------------------------------------------------
df = pd.read_csv(PANEL_FILE, parse_dates=["date"])

# --------------------------------------------------
# Basic cleaning
# --------------------------------------------------
print("Before cleaning:", len(df))

# remove duplicates
df = df.drop_duplicates(subset=["symbol", "date"]).copy()

# remove impossible rows
df = df[
    (df["close"] > 0) &
    (df["open"] > 0) &
    (df["high"] > 0) &
    (df["low"] > 0) &
    (df["volume"] >= 0)
].copy()

df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

print("After cleaning:", len(df))

Before cleaning: 61490
After cleaning: 61489


### Compute Returns & Volatility

In [22]:
# --------------------------------------------------
# Returns and rolling volatility
# --------------------------------------------------
def add_returns_and_volatility(data: pd.DataFrame, vol_lookback: int = 20) -> pd.DataFrame:
    out = data.copy()

    # log return
    out["log_ret"] = (
        out.groupby("symbol")["close"]
        .transform(lambda s: np.log(s).diff())
    )

    # rolling realized volatility of daily log returns
    out["daily_vol"] = (
        out.groupby("symbol")["log_ret"]
        .transform(lambda s: s.rolling(vol_lookback, min_periods=vol_lookback).std())
    )

    return out

df = add_returns_and_volatility(df, vol_lookback=VOL_LOOKBACK)
print(df.head(30))

         date symbol  open  high   low  close   volume   log_ret  daily_vol
0  2018-07-02    ACB  6.87  6.87  6.33   6.43  6858520       NaN        NaN
1  2018-07-03    ACB  6.45  6.56  5.91   5.98  7375018 -0.072554        NaN
2  2018-07-04    ACB  5.89  6.18  5.89   6.18  4127586  0.032898        NaN
3  2018-07-05    ACB  6.04  6.18  5.69   5.77  4948503 -0.068646        NaN
4  2018-07-06    ACB  5.60  6.33  5.60   6.33  7854475  0.092628        NaN
5  2018-07-09    ACB  6.37  6.58  6.33   6.37  4587386  0.006299        NaN
6  2018-07-10    ACB  6.37  6.60  6.35   6.54  3244193  0.026338        NaN
7  2018-07-11    ACB  6.45  6.47  5.98   6.16  7023863 -0.059860        NaN
8  2018-07-12    ACB  6.76  6.76  6.16   6.39  4150902  0.036657        NaN
9  2018-07-13    ACB  6.39  6.66  5.98   6.60  3940996  0.032335        NaN
10 2018-07-16    ACB  6.66  6.79  6.60   6.66  3769038  0.009050        NaN
11 2018-07-17    ACB  6.93  6.93  6.54   6.87  5135363  0.031045        NaN
12 2018-07-1

### Triple-Barrier Labeling

In [23]:
# --------------------------------------------------
# Triple-barrier core
# --------------------------------------------------
def triple_barrier_for_symbol(
    g: pd.DataFrame,
    pt_mult: float = 2.0,
    sl_mult: float = 2.0,
    horizon: int = 10
) -> pd.DataFrame:
    """
    Apply triple-barrier labeling to one symbol.

    Required columns in g:
    date, symbol, close, high, low, daily_vol
    """
    n = len(g)

    labels = np.full(n, np.nan)
    hit_dates = [pd.NaT] * n
    hit_types = [None] * n
    vertical_dates = [pd.NaT] * n
    upper_barriers = np.full(n, np.nan)
    lower_barriers = np.full(n, np.nan)
    event_returns = np.full(n, np.nan)

    close_arr = g["close"].to_numpy()
    high_arr = g["high"].to_numpy()
    low_arr = g["low"].to_numpy()
    vol_arr = g["daily_vol"].to_numpy()
    date_arr = g["date"].to_numpy()

    for i in range(n):
        vol = vol_arr[i]

        # skip if volatility unavailable
        if np.isnan(vol):
            continue

        # vertical barrier index
        j_end = min(i + horizon, n - 1)
        if j_end <= i:
            continue

        entry = close_arr[i]
        upper = entry * np.exp(pt_mult * vol)
        lower = entry * np.exp(-sl_mult * vol)

        upper_barriers[i] = upper
        lower_barriers[i] = lower
        vertical_dates[i] = pd.Timestamp(date_arr[j_end])

        label = 0
        hit_idx = j_end
        hit_type = "vb"

        # scan forward to find first touch
        for j in range(i + 1, j_end + 1):
            up_hit = high_arr[j] >= upper
            dn_hit = low_arr[j] <= lower

            if up_hit and dn_hit:
                # ambiguous same-bar touch in daily OHLC data
                # conservative rule: compare close-to-bar distances using that day's open/close not available reliably
                # choose the barrier with smaller log distance from entry in absolute terms; if equal -> 0
                up_dist = np.log(upper / entry)
                dn_dist = abs(np.log(lower / entry))
                if up_dist < dn_dist:
                    label = 1
                    hit_idx = j
                    hit_type = "pt"
                elif dn_dist < up_dist:
                    label = -1
                    hit_idx = j
                    hit_type = "sl"
                else:
                    label = 0
                    hit_idx = j
                    hit_type = "ambiguous"
                break
            elif up_hit:
                label = 1
                hit_idx = j
                hit_type = "pt"
                break
            elif dn_hit:
                label = -1
                hit_idx = j
                hit_type = "sl"
                break

        labels[i] = label
        hit_dates[i] = pd.Timestamp(date_arr[hit_idx])
        hit_types[i] = hit_type
        event_returns[i] = np.log(close_arr[hit_idx] / entry)

    out = g.copy()
    out["tb_label"] = labels
    out["tb_hit_date"] = hit_dates
    out["tb_hit_type"] = hit_types
    out["tb_vertical_date"] = vertical_dates
    out["tb_upper_barrier"] = upper_barriers
    out["tb_lower_barrier"] = lower_barriers
    out["tb_event_log_return"] = event_returns

    return out

In [24]:
# --------------------------------------------------
# Apply to all symbols
# --------------------------------------------------
labeled_list = []

for symbol, g in df.groupby("symbol", sort=True):
    print(f"Labeling {symbol} ...")
    labeled_g = triple_barrier_for_symbol(
        g,
        pt_mult=PT_MULT,
        sl_mult=SL_MULT,
        horizon=VERTICAL_HORIZON
    )
    labeled_list.append(labeled_g)

labeled = pd.concat(labeled_list, axis=0, ignore_index=True)
labeled = labeled.sort_values(["symbol", "date"]).reset_index(drop=True)

print(labeled.head(30))

Labeling ACB ...
Labeling BID ...
Labeling BVH ...
Labeling CTG ...
Labeling DGC ...
Labeling FPT ...
Labeling GAS ...
Labeling GVR ...
Labeling HDB ...
Labeling HPG ...
Labeling HSG ...
Labeling LPB ...
Labeling MBB ...
Labeling MSN ...
Labeling MWG ...
Labeling PLX ...
Labeling PNJ ...
Labeling POW ...
Labeling SAB ...
Labeling SHB ...
Labeling SSI ...
Labeling STB ...
Labeling TCB ...
Labeling TPB ...
Labeling VCB ...
Labeling VHM ...
Labeling VIB ...
Labeling VIC ...
Labeling VJC ...
Labeling VNM ...
Labeling VPB ...
Labeling VRE ...
         date symbol  open  high   low  close   volume   log_ret  daily_vol  \
0  2018-07-02    ACB  6.87  6.87  6.33   6.43  6858520       NaN        NaN   
1  2018-07-03    ACB  6.45  6.56  5.91   5.98  7375018 -0.072554        NaN   
2  2018-07-04    ACB  5.89  6.18  5.89   6.18  4127586  0.032898        NaN   
3  2018-07-05    ACB  6.04  6.18  5.69   5.77  4948503 -0.068646        NaN   
4  2018-07-06    ACB  5.60  6.33  5.60   6.33  7854475  0.092

## 2. Feature Engineering

### Feature Construction

In [ ]:
def make_features(
    df: pd.DataFrame,
    windows: tuple = (5, 10, 20),
    lags: tuple = (1, 2, 3, 5, 10, 20),
    add_technical: bool = True,
) -> pd.DataFrame:
    """
    Feature engineering for daily stock panel data.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain at least:
        ["date", "symbol", open, high, low, "close", "volume"]
    windows : tuple
        Rolling window sizes.
    lags : tuple
        Lag periods for return features.
    add_technical : bool
        Whether to add RSI / MACD / ATR features.

    Returns
    -------
    pd.DataFrame
        Original dataframe with added features.
    """

    required_cols = ["date", "symbol", "open", "high", "low", "close", "volume", "log_ret", "daily_vol"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    out = out.sort_values(["symbol", "date"]).reset_index(drop=True)

    # -----------------------------
    # Basic returns
    # -----------------------------
    out["ret_1"] = out.groupby("symbol")["close"].pct_change()

    # lagged log returns
    for lag in lags:
        out[f"log_ret_lag_{lag}"] = out.groupby("symbol")["close"].transform(
            lambda s, lag=lag: np.log(s / s.shift(lag))
        )

    # -----------------------------
    # Price trend / momentum
    # -----------------------------
    for win in windows:
        rolling_mean = out.groupby("symbol")["close"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).mean()
        )
        rolling_std = out.groupby("symbol")["close"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).std()
        )

        out[f"ma_{win}"] = rolling_mean
        out[f"std_price_{win}"] = rolling_std
        out[f"price_to_ma_{win}"] = out["close"] / rolling_mean
        out[f"zscore_price_{win}"] = (out["close"] - rolling_mean) / rolling_std

        # momentum over window
        out[f"mom_{win}"] = out.groupby("symbol")["close"].transform(
            lambda s, win=win: s / s.shift(win) - 1
        )

    # -----------------------------
    # Volatility features
    # -----------------------------
    for win in windows:
        out[f"volatility_{win}"] = out.groupby("symbol")["log_ret"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).std()
        )

        out[f"realized_var_{win}"] = out.groupby("symbol")["log_ret"].transform(
            lambda s, win=win: (s ** 2).rolling(win, min_periods=win).mean()
        )

    # -----------------------------
    # Range / candle features
    # -----------------------------
    out["hl_range"] = (out["high"] - out["low"]) / out["close"]
    out["oc_return"] = (out["close"] - out["open"]) / out["open"]
    out["co_gap"] = out.groupby("symbol")["open"].transform(
        lambda s: s / s.shift(1) - 1
    )

    # previous close gap
    prev_close = out.groupby("symbol")["close"].shift(1)
    out["open_to_prev_close"] = out["open"] / prev_close - 1

    # -----------------------------
    # Volume features
    # -----------------------------
    out["vol_chg_1"] = out.groupby("symbol")["volume"].pct_change()

    # log volume is often more stable
    out["log_volume"] = np.log1p(out["volume"])

    for win in windows:
        vol_ma = out.groupby("symbol")["volume"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).mean()
        )
        vol_std = out.groupby("symbol")["volume"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).std()
        )

        out[f"vol_ma_{win}"] = vol_ma
        out[f"vol_std_{win}"] = vol_std
        out[f"rel_volume_{win}"] = out["volume"] / vol_ma
        out[f"zscore_volume_{win}"] = (out["volume"] - vol_ma) / vol_std

    # -----------------------------
    # Technical indicators
    # -----------------------------
    if add_technical:
        # RSI(14)
        def _rsi(series: pd.Series, period: int = 14) -> pd.Series:
            delta = series.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)

            avg_gain = gain.rolling(period, min_periods=period).mean()
            avg_loss = loss.rolling(period, min_periods=period).mean()

            rs = avg_gain / avg_loss.replace(0, np.nan)
            rsi = 100 - (100 / (1 + rs))
            return rsi

        out["rsi_14"] = out.groupby("symbol")["close"].transform(_rsi)

        # MACD(12,26,9)
        def _ema(series: pd.Series, span: int) -> pd.Series:
            return series.ewm(span=span, adjust=False).mean()

        ema12 = out.groupby("symbol")["close"].transform(lambda s: _ema(s, 12))
        ema26 = out.groupby("symbol")["close"].transform(lambda s: _ema(s, 26))

        out["macd"] = ema12 - ema26
        out["macd_signal"] = out.groupby("symbol")["macd"].transform(
            lambda s: s.ewm(span=9, adjust=False).mean()
        )
        out["macd_hist"] = out["macd"] - out["macd_signal"]

        # ATR(14)
        def _atr(g: pd.DataFrame, period: int = 14) -> pd.Series:
            prev_close = g["close"].shift(1)
            tr1 = g["high"] - g["low"]
            tr2 = (g["high"] - prev_close).abs()
            tr3 = (g["low"] - prev_close).abs()
            tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
            atr = tr.rolling(period, min_periods=period).mean()
            return atr

        out["atr_14"] = (
            out.groupby("symbol", group_keys=False)
            .apply(_atr)
            .reset_index(level=0, drop=True)
        )

        out["atr_pct_14"] = out["atr_14"] / out["close"]

    # -----------------------------
    # Calendar features
    # -----------------------------
    out["day_of_week"] = out["date"].dt.dayofweek
    out["month"] = out["date"].dt.month
    out["quarter"] = out["date"].dt.quarter

    # -----------------------------
    # Replace infs
    # -----------------------------
    out = out.replace([np.inf, -np.inf], np.nan)

    return out

In [28]:
df_feat = make_features(labeled)
print(df_feat.columns.tolist())
print(df_feat.head())

['date', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'log_ret', 'daily_vol', 'tb_label', 'tb_hit_date', 'tb_hit_type', 'tb_vertical_date', 'tb_upper_barrier', 'tb_lower_barrier', 'tb_event_log_return', 'ret_1', 'log_ret_lag_1', 'log_ret_lag_2', 'log_ret_lag_3', 'log_ret_lag_5', 'log_ret_lag_10', 'log_ret_lag_20', 'ma_5', 'std_price_5', 'price_to_ma_5', 'zscore_price_5', 'mom_5', 'ma_10', 'std_price_10', 'price_to_ma_10', 'zscore_price_10', 'mom_10', 'ma_20', 'std_price_20', 'price_to_ma_20', 'zscore_price_20', 'mom_20', 'volatility_5', 'realized_var_5', 'volatility_10', 'realized_var_10', 'volatility_20', 'realized_var_20', 'hl_range', 'oc_return', 'co_gap', 'open_to_prev_close', 'vol_chg_1', 'log_volume', 'vol_ma_5', 'vol_std_5', 'rel_volume_5', 'zscore_volume_5', 'vol_ma_10', 'vol_std_10', 'rel_volume_10', 'zscore_volume_10', 'vol_ma_20', 'vol_std_20', 'rel_volume_20', 'zscore_volume_20', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'atr_14', 'atr_pct_14', 'day_of_week', 'mo

C:\Users\T14s G4\AppData\Local\Temp\ipykernel_28104\3796191266.py:159: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_atr)


### Feature Selection

In [29]:
exclude_cols = {
    "date", "symbol",
    "tb_label", "tb_hit_date", "tb_hit_type",
    "tb_vertical_date", "tb_upper_barrier", "tb_lower_barrier",
    "tb_event_log_return"
}

feature_cols = [c for c in df_feat.columns if c not in exclude_cols]

In [31]:
df_model = df_feat.dropna(subset=feature_cols + ["tb_label"]).copy()
df_model.head()

,date,symbol,open,high,low,close,volume,log_ret,daily_vol,tb_label,...,zscore_volume_20,rsi_14,macd,macd_signal,macd_hist,atr_14,atr_pct_14,day_of_week,month,quarter
20,2018-07-30,ACB,6.89,7.05,6.87,6.97,3446870,0.014451,0.041665,0.0,...,-1.098008,58.301158,0.127770,0.105217,0.022553,0.443571,0.063640,0,7,3
21,2018-07-31,ACB,6.95,7.05,6.87,6.91,4169999,-0.008646,0.037749,0.0,...,-0.565693,66.519824,0.130073,0.110188,0.019885,0.416429,0.060265,1,7,3
22,2018-08-01,ACB,6.91,6.97,6.78,6.85,3370615,-0.008721,0.037405,1.0,...,-1.047611,60.952381,0.125609,0.113272,0.012337,0.387143,0.056517,2,8,3
23,2018-08-02,ACB,6.85,6.87,6.70,6.85,4547726,0.000000,0.033189,1.0,...,-0.270564,56.613757,0.120680,0.114754,0.005926,0.350714,0.051199,3,8,3
24,2018-08-03,ACB,6.91,7.03,6.85,6.85,3765836,0.000000,0.026665,1.0,...,-0.707588,55.191257,0.115444,0.114892,0.000552,0.350000,0.051095,4,8,3


In [34]:
# --------------------------------------------------
# Save labeled dataset
# --------------------------------------------------
df_model.to_csv(PROCESSED_OUT, index=False)
print("Saved:", PROCESSED_OUT)

Saved: ./data\processed_vn30_panel_daily.csv


## 3. Data Splitting

### Purged Time Series Cross-Validation

In [ ]:
class PurgedTimeSeriesSplit:
    def __init__(
        self,
        n_splits=5,
        embargo_days=10,
        min_train_size=252
    ):
        self.n_splits = n_splits
        self.embargo_days = embargo_days
        self.min_train_size = min_train_size

    def split(self, df):
        df = df.sort_values("date").reset_index(drop=True).copy()
        unique_dates = np.array(sorted(df["date"].unique()))

        n_dates = len(unique_dates)
        fold_size = n_dates // (self.n_splits + 1)

        for fold in range(self.n_splits):
            val_start_idx = (fold + 1) * fold_size
            if fold == self.n_splits - 1:
                val_end_idx = n_dates - 1
            else:
                val_end_idx = (fold + 2) * fold_size - 1

            val_start_date = unique_dates[val_start_idx]
            val_end_date = unique_dates[val_end_idx]
            embargo_end_date = pd.Timestamp(val_end_date) + pd.Timedelta(days=self.embargo_days)

            val_mask = (df["date"] >= val_start_date) & (df["date"] <= val_end_date)

            train_mask = df["date"] < val_start_date

            # purge training rows whose event overlaps validation
            overlap_mask = train_mask & (df["tb_hit_date"] >= val_start_date)
            train_mask = train_mask & (~overlap_mask)

            # embargo after validation
            embargo_mask = (df["date"] > val_end_date) & (df["date"] <= embargo_end_date)
            train_mask = train_mask & (~embargo_mask)

            train_idx = np.where(train_mask)[0]
            val_idx = np.where(val_mask)[0]

            if len(train_idx) < self.min_train_size:
                continue
            if len(val_idx) == 0:
                continue

            yield train_idx, val_idx

### Time-series Train - Validation / Test Split

In [ ]:
def chronological_holdout_split(df, test_start="2025-01-01"):
    trainval = df[df["date"] < pd.to_datetime(test_start)].copy()
    test = df[df["date"] >= pd.to_datetime(test_start)].copy()
    return trainval, test

In [40]:
trainval_df, test_df = chronological_holdout_split(df_model, test_start="2025-01-01")

In [41]:
print("trainval:", trainval_df["date"].min(), "->", trainval_df["date"].max(), len(trainval_df))
print("test    :", test_df["date"].min(), "->", test_df["date"].max(), len(test_df))

trainval: 2018-07-30 00:00:00 -> 2024-12-31 00:00:00 51372
test    : 2025-01-02 00:00:00 -> 2026-03-12 00:00:00 9365


## 4. Model Training

### Hyperparameter Tuning on Train/Validation
- Tune 5 candidate versions for each ML model on the train-validation split only.
- Select the best version using validation performance, then evaluate only the selected model on the test set.

In [ ]:
# --------------------------------------------------
# 5 candidate versions for each ML family
# --------------------------------------------------
random_state = 42

model_families = {
    "xgb": [
        XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=random_state, n_jobs=-1,
        ),
        XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=random_state, n_jobs=-1,
        ),
        XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=random_state, n_jobs=-1,
        ),
        XGBClassifier(
            n_estimators=400, max_depth=5, learning_rate=0.03,
            subsample=0.9, colsample_bytree=0.9,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=random_state, n_jobs=-1,
        ),
        XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.02,
            subsample=0.8, colsample_bytree=0.8,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=random_state, n_jobs=-1,
        ),
    ],

    "rf": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=100, max_depth=5, min_samples_leaf=10,
                class_weight="balanced_subsample",
                random_state=random_state, n_jobs=-1
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=200, max_depth=8, min_samples_leaf=10,
                class_weight="balanced_subsample",
                random_state=random_state, n_jobs=-1
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=300, max_depth=10, min_samples_leaf=10,
                class_weight="balanced_subsample",
                random_state=random_state, n_jobs=-1
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=400, max_depth=None, min_samples_leaf=5,
                class_weight="balanced_subsample",
                random_state=random_state, n_jobs=-1
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=500, max_depth=None, min_samples_leaf=3,
                class_weight="balanced_subsample",
                random_state=random_state, n_jobs=-1
            ))
        ]),
    ],

    "svm": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(C=0.3, kernel="rbf", gamma="scale", class_weight="balanced"))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(C=1.0, kernel="rbf", gamma="scale", class_weight="balanced"))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(C=3.0, kernel="rbf", gamma="scale", class_weight="balanced"))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(C=1.0, kernel="linear", class_weight="balanced"))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(C=3.0, kernel="linear", class_weight="balanced"))
        ]),
    ],

    "mlp": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(64,),
                activation="relu",
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                early_stopping=True,
                random_state=random_state,
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(128,),
                activation="relu",
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                early_stopping=True,
                random_state=random_state,
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                early_stopping=True,
                random_state=random_state,
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(256, 128),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=5e-4,
                max_iter=300,
                early_stopping=True,
                random_state=random_state,
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(256, 128, 64),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=5e-4,
                max_iter=300,
                early_stopping=True,
                random_state=random_state,
            ))
        ]),
    ],
}

In [51]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [48]:
def encode_labels(y_train, y_other=None):
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)

    if y_other is None:
        return y_train_enc, le

    y_other_enc = le.transform(y_other)
    return y_train_enc, y_other_enc, le

In [ ]:
def run_purged_cv(
    df,
    models: Dict[str, object],
    feature_cols,
    n_splits=5,
    embargo_days=10,
    min_train_size=252,
):
    df = df.sort_values(["date", "symbol"]).reset_index(drop=True).copy()

    cv = PurgedTimeSeriesSplit(
        n_splits=n_splits,
        embargo_days=embargo_days,
        min_train_size=min_train_size,
    )
    rows = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(df), start=1):
        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()

        X_train = train_df[feature_cols]
        y_train = train_df["tb_label"]

        X_val = val_df[feature_cols]
        y_val = val_df["tb_label"]

        print(f"\nFold {fold}: train={len(train_df):,}, val={len(val_df):,}, "
              f"train_end={train_df["date"].max().date()}, val=[{val_df["date"].min().date()} -> {val_df["date"].max().date()}]")

        for model_name, model in models.items():
            clf = clone(model)

            if model_name == "xgb":
                y_train_enc, y_val_enc, le = encode_labels(y_train, y_val)
                clf.fit(X_train, y_train_enc)
                pred_enc = clf.predict(X_val)
                y_pred = le.inverse_transform(pred_enc.astype(int))
            else:
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_val)

            metrics = compute_metrics(y_val, y_pred)

            row = {
                "fold": fold,
                "model": model_name,
                "n_train": len(train_df),
                "n_val": len(val_df),
                **metrics
            }
            rows.append(row)

    cv_results = pd.DataFrame(rows)
    return cv_results

In [ ]:
# --------------------------------------------------
# Tune all versions 
# --------------------------------------------------
all_cv_results = []

for family_name, candidates in model_families.items():
    for version_id, candidate in enumerate(candidates, start=1):
        print(f"\n{'='*80}")
        print(f"Tuning {family_name} - version {version_id}")
        print(f"{'='*80}")

        one_model_dict = {family_name: candidate}

        cv_res = run_purged_cv(
            trainval_df,
            models=one_model_dict,
            feature_cols=feature_cols,
            n_splits=5,
            embargo_days=10,
            min_train_size=252 * 2,
        ).copy()

        cv_res["family"] = family_name
        cv_res["version"] = version_id
        all_cv_results.append(cv_res)

all_cv_results = pd.concat(all_cv_results, ignore_index=True)

In [ ]:
all_cv_results

,fold,model,n_train,n_val,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_precision,macro_recall
0,1,xgb,8376,8544,0.407186,0.385383,0.382262,0.407777,0.379823,0.385383
1,1,rf,8376,8544,0.380501,0.425589,0.374586,0.387467,0.384300,0.425589
2,1,svm,8376,8544,0.380735,0.405215,0.366796,0.380927,0.375221,0.405215
3,1,mlp,8376,8544,0.426264,0.403027,0.396919,0.429171,0.393558,0.403027
4,2,xgb,16877,8545,0.395670,0.373229,0.366319,0.391272,0.403408,0.373229
5,2,rf,16877,8545,0.355061,0.414298,0.351010,0.353179,0.368479,0.414298
6,2,svm,16877,8545,0.362902,0.400316,0.351998,0.366906,0.365798,0.400316
7,2,mlp,16877,8545,0.399766,0.400386,0.386565,0.404550,0.385964,0.400386
8,3,xgb,25461,8576,0.410331,0.369741,0.359702,0.397065,0.385157,0.369741
9,3,rf,25461,8576,0.385028,0.431381,0.369849,0.397921,0.379553,0.431381


In [ ]:
# --------------------------------------------------
# Average CV metrics for each family/version
# --------------------------------------------------
tuning_summary = (
    all_cv_results
    .groupby(["family", "version"], as_index=False)[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1",
         "macro_precision", "macro_recall"]
    ]
    .mean()
    .sort_values(["family", "macro_f1"], ascending=[True, False])
)

tuning_summary

Best model from CV: svm


In [ ]:
# --------------------------------------------------
# Best version for each model family
# --------------------------------------------------
best_versions = (
    tuning_summary
    .sort_values(["family", "macro_f1"], ascending=[True, False])
    .groupby("family", as_index=False)
    .first()
)

best_versions

In [ ]:
# --------------------------------------------------
# Baseline models
# --------------------------------------------------
class ModeBaseline:
    def fit(self, X, y):
        y = pd.Series(y)
        self.mode_label_ = y.mode().iloc[0]
        return self

    def predict(self, X):
        return np.repeat(self.mode_label_, len(X))


class RandomBaseline:
    def __init__(self, random_state=42):
        self.random_state = random_state

    def fit(self, X, y):
        y = pd.Series(y)
        probs = y.value_counts(normalize=True).sort_index()
        self.labels_ = probs.index.to_numpy()
        self.probs_ = probs.values
        self.rng_ = np.random.default_rng(self.random_state)
        return self

    def predict(self, X):
        return self.rng_.choice(self.labels_, size=len(X), p=self.probs_)


class PrevLabelBaseline:
    """
    Previous observed label by symbol:
    - first test row of a symbol uses the last trainval label for that symbol
    - next test rows use previous test-row prediction input logic based on chronological order
    """
    def fit(self, train_df):
        last_lbl = (
            train_df.sort_values(["symbol", "date"])
            .groupby("symbol")["tb_label"]
            .last()
        )
        self.last_train_label_by_symbol_ = last_lbl.to_dict()
        self.global_mode_ = train_df["tb_label"].mode().iloc[0]
        return self

    def predict_from_df(self, df):
        df = df.sort_values(["symbol", "date"]).copy()
        preds = []

        for symbol, g in df.groupby("symbol", sort=False):
            prev = self.last_train_label_by_symbol_.get(symbol, self.global_mode_)
            for _ in range(len(g)):
                preds.append(prev)
                # naive persistence: keep using previous observed/persisted label
                prev = prev

        out = pd.Series(preds, index=df.index)
        out = out.reindex(df.index)
        return out.loc[df.index].values


# fit baselines
mode_baseline = ModeBaseline().fit(X_trainval, y_train)
random_baseline = RandomBaseline(random_state=42).fit(X_trainval, y_train)
prev_baseline = PrevLabelBaseline().fit(trainval_df)

## 5.Final evaluation

In [ ]:
def evaluate(
    clf,
    test_df,
    feature_cols,
):
    X_test = test_df[feature_cols]
    y_test = test_df["tb_label"]

    if clf.__class__.__name__ == "XGBClassifier":
        y_train_enc, y_test_enc, le = encode_labels(y_train, y_test)
        pred_enc = clf.predict(X_test)
        y_pred = le.inverse_transform(pred_enc.astype(int))
    else:
        y_pred = clf.predict(X_test)

    metrics = compute_metrics(y_test, y_pred)

    print("\nFinal test metrics:")
    print(metrics)

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    labels_sorted = sorted(trainval_df["tb_label"].unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

    print("\nConfusion matrix (labels order =", labels_sorted, "):")
    print(cm)

    return {
        "model": clf,
        "metrics": metrics,
        "y_test": y_test,
        "y_pred": y_pred,
        "confusion_matrix": cm,
    }

In [ ]:
# --------------------------------------------------
# Fit best ML models on full trainval
# --------------------------------------------------
X_trainval = trainval_df[feature_cols]
y_train = trainval_df["tb_label"]   # IMPORTANT: create global y_train for your evaluate(...)

best_ml_models = {}

for _, row in best_versions.iterrows():
    family = row["family"]
    version = int(row["version"])
    clf = clone(model_families[family][version - 1])

    if family == "xgb":
        y_train_enc, le = encode_labels(y_train)
        clf.fit(X_trainval, y_train_enc)
    else:
        clf.fit(X_trainval, y_train)

    best_ml_models[f"{family}_v{version}"] = clf

list(best_ml_models.keys())

In [ ]:
# --------------------------------------------------
# Evaluate best ML models with your existing evaluate(...)
# --------------------------------------------------
ml_test_results = []

for model_name, clf in best_ml_models.items():
    best_model_name = model_name   # IMPORTANT: create global expected by evaluate(...)
    out = evaluate(
        clf=clf,
        test_df=test_df,
        feature_cols=feature_cols,
    )

    ml_test_results.append({
        "name": model_name,
        "type": "ml",
        **out["metrics"],
    })

ml_test_results = pd.DataFrame(ml_test_results)
ml_test_results

In [ ]:
# --------------------------------------------------
# Evaluate baselines on test
# --------------------------------------------------
y_test = test_df["tb_label"]

baseline_rows = []

# Mode baseline
y_pred_mode = mode_baseline.predict(test_df[feature_cols])
baseline_rows.append({
    "name": "mode_baseline",
    "type": "baseline",
    **compute_metrics(y_test, y_pred_mode),
})

# Random baseline
y_pred_random = random_baseline.predict(test_df[feature_cols])
baseline_rows.append({
    "name": "random_baseline",
    "type": "baseline",
    **compute_metrics(y_test, y_pred_random),
})

# Previous-label baseline
y_pred_prev = prev_baseline.predict_from_df(test_df)
baseline_rows.append({
    "name": "prev_label_baseline",
    "type": "baseline",
    **compute_metrics(y_test, y_pred_prev),
})

baseline_test_results = pd.DataFrame(baseline_rows)
baseline_test_results

In [ ]:
# --------------------------------------------------
# Final comparison: best ML models vs 3 baselines
# --------------------------------------------------
final_comparison = pd.concat(
    [ml_test_results, baseline_test_results],
    ignore_index=True
).sort_values(["macro_f1", "balanced_accuracy"], ascending=False)

final_comparison